# Compact conditioning keys

One worked example of the selected layout: compact glyphs and values inside a 3 × 4 grid, with one shared key. R01 is a real MOJAVE observation. S01–S11 are synthetic examples.

- **Proposed plotting code** owns the glyph drawings, the tile keys, and their layout. It uses the existing shared image, overlay, and colorbar functions.
- **Caller code** chooses which fields and series to show, supplies their labels and styles, and gives the plotter drawable marks plus display values. The proposed code has no dependency on the callback or MOJAVE metadata.

The glyphs are defined in this notebook prototype. The next step would be to move the generic rendering code into the shared plotting module.

In [ ]:
from dataclasses import dataclass
from io import BytesIO

import matplotlib as mpl
import numpy as np
from IPython.display import Image, display
from matplotlib.collections import LineCollection
from matplotlib.colors import to_rgba
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnchoredOffsetbox, DrawingArea, HPacker, TextArea, VPacker
from matplotlib.patches import PathPatch
from matplotlib.path import Path as MarkerPath

from radio_diffusion.utils import plotting
from radio_diffusion.utils.plotting import ImageOverlay

## What would go into plotting.py

The grid receives the same inputs as the existing plotter, plus optional keys for the individual tiles. Each entry records the glyph type, its meaning and value, and its appearance. Its meaning is supplied by the caller.

In [ ]:
@dataclass(frozen=True)
class SeriesStyle:
    color: str
    linestyle: str = "solid"
    label: str | None = None


@dataclass(frozen=True)
class KeyEntry:
    kind: str
    meaning: str
    value: str
    style: SeriesStyle
    faded: bool = False


GLYPH_PATHS = {
    "arrow": [(2, 5), (15, 5), (11, 8), (15, 5), (11, 2)],
    "arc": [(x, 8 - 0.065 * (x - 9)**2) for x in np.linspace(2, 16, 20)],
    "polyline": [(2, 3), (6, 7), (11, 4), (16, 7)],
}


def glyph(entry):
    points = np.asarray(GLYPH_PATHS[entry.kind])
    area = DrawingArea(18, 10, 0, 0)
    if entry.faded:
        segments = np.stack((points[:-1], points[1:]), axis=1)
        colors = np.tile(to_rgba(entry.style.color), (len(segments), 1))
        colors[:, 3] *= (1 + np.cos(np.linspace(-np.pi, np.pi, len(segments)))) / 2
        mark = LineCollection(segments, colors=colors, linewidths=1.25)
    else:
        linestyle = entry.style.linestyle
        mark = PathPatch(
            MarkerPath(points), fill=False, edgecolor=entry.style.color, linewidth=1.25,
            linestyle=(0, (2, 1.2)) if linestyle in ("dashed", "--") else linestyle,
        )
    area.add_artist(mark)
    return area


def add_tile_key(ax, entries):
    if not entries:
        return
    rows = {}
    for entry in entries:
        value = TextArea(entry.value, textprops={"color": "white", "fontsize": 8})
        token = HPacker(children=[glyph(entry), value], align="center", pad=0, sep=2)
        rows.setdefault(entry.style, []).append(token)
    children = [HPacker(children=row, align="center", pad=0, sep=6) for row in rows.values()]
    box = AnchoredOffsetbox(
        loc="lower right", child=VPacker(children=children, align="left", pad=0, sep=2),
        frameon=True, pad=0.4, borderpad=0.55,
    )
    box.patch.set(facecolor="#172031", edgecolor="#99a5b6", linewidth=0.55, alpha=0.92)
    ax.add_artist(box)


def add_shared_key(figure, tile_entries):
    entries = [entry for tile in tile_entries for entry in tile]
    handles = []
    for kind, meaning in dict.fromkeys((entry.kind, entry.meaning) for entry in entries):
        marker = MarkerPath(np.asarray(GLYPH_PATHS[kind]) - (9, 5))
        handles.append(Line2D(
            [], [], marker=marker, markersize=12, markerfacecolor="none",
            markeredgecolor="#42536a", linestyle="none", label=meaning,
        ))
    for style in dict.fromkeys(entry.style for entry in entries):
        if style.label:
            handles.append(Line2D(
                [0, 1], [0, 0], linewidth=1.4,
                color=style.color, linestyle=style.linestyle, label=style.label,
            ))
    if handles:
        figure.legend(
            handles=handles, loc="outside lower center", ncols=len(handles),
            frameon=False, handlelength=2.6,
        )


def plot_keyed_grid(images, *, tile_entries=None, view_limits=None, figsize=None, **grid_options):
    if tile_entries is not None and len(tile_entries) != len(images):
        raise ValueError("Supply one key per image.")
    figure, axes = plotting.plot_grid(images, **grid_options)
    if figsize is not None:
        figure.set_size_inches(*figsize)
    if view_limits is not None:
        xlim, ylim = view_limits
        for ax in list(axes.flat)[:len(images)]:
            ax.set(xlim=xlim, ylim=ylim)
    if tile_entries is not None:
        for ax, entries in zip(axes.flat, tile_entries):
            add_tile_key(ax, entries)
        add_shared_key(figure, tile_entries)
    return figure, axes

## Example data

These two cells build the demo batch. The training callback already has the generated images, conditioning values, and validity masks. A false validity mask would become an unavailable value before building a display entry.

In [ ]:
samples = [
    dict(id="S01", source="J0813+2910", image_pa=121, image_extent=2.85, peak=0.44,
         annotations={"conditioning": (106, 2.55), "estimate": (121, 2.85)}),
    dict(id="S02", source="J1126−0427", image_pa=66, image_extent=1.60, peak=0.18,
         annotations={"conditioning": (66, 1.60)}),
    dict(id="S03", source="J1502+3741", image_pa=295, image_extent=2.40, peak=0.56,
         annotations={"estimate": (295, 2.40)}),
    dict(id="S04", source="J2034−1152", image_pa=205, image_extent=2.25, peak=0.29,
         annotations={"conditioning": (193, None), "estimate": (205, 2.25)}),
]

SIZE = 88
REFERENCE = (SIZE - 1) / 2
PIXEL_SCALE_MAS = 0.086
coordinates = plotting.mojave_coordinates({
    "reference_pixel_xy": [REFERENCE, REFERENCE],
    "pixel_to_east_north_mas": [[-PIXEL_SCALE_MAS, 0], [0, PIXEL_SCALE_MAS]],
})


def demo_image(sample, seed):
    row, column = np.indices((SIZE, SIZE))
    east, north = coordinates.from_pixels(np.stack((column, row), axis=-1)).transpose(2, 0, 1)
    pa = np.deg2rad(sample["image_pa"])
    along = east * np.sin(pa) + north * np.cos(pa)
    across = east * np.cos(pa) - north * np.sin(pa)
    extent = sample["image_extent"]
    forward = np.maximum(along, 0)
    bend = 0.035 * forward**2
    width = 0.105 + 0.045 * forward
    core = np.exp(-0.5 * ((along / 0.20)**2 + (across / 0.14)**2))
    jet = 0.28 * np.exp(-forward / 1.5 - 0.5 * ((across - bend) / width)**2)
    jet *= (along >= 0) * np.clip((extent - along) / 0.42, 0, 1)
    knot = 0.095 * np.exp(
        -0.5 * (((along - extent * 0.63) / 0.22)**2 + ((across - bend) / 0.14)**2)
    )
    noise = np.random.default_rng(seed).normal(0, 0.00025, core.shape)
    return sample["peak"] * (core + jet + knot) + noise


for index in range(5, 12):
    pa = (40 + index * 37) % 360
    extent = 1.45 + 0.19 * (index % 8)
    samples.append(dict(
        id=f"S{index:02d}", source=f"Synthetic {index:02d}",
        image_pa=pa, image_extent=extent, peak=0.17 + 0.07 * (index % 5),
        annotations={"conditioning": ((pa - 11) % 360, extent + 0.25), "estimate": (pa, extent)},
    ))


def synthetic_ridge(sample):
    along = np.linspace(0.15, sample["image_extent"], 17)
    pa = np.deg2rad(sample["image_pa"])
    forward = np.array([np.sin(pa), np.cos(pa)])
    across = np.array([np.cos(pa), -np.sin(pa)])
    return along[:, None] * forward + (0.035 * along**2)[:, None] * across


synthetic_samples = samples
for index, sample in enumerate(synthetic_samples):
    sample["image"] = demo_image(sample, seed=40 + index)
    sample["coordinates"] = coordinates
    sample["ridge"] = synthetic_ridge(sample)

In [ ]:
import warnings

from astropy.io import fits
from astropy.wcs import FITSFixedWarning

from radio_diffusion.data.mojave_metadata import (
    JetOrientationSettings, crop_spatial_geometry, get_noise_jy_per_beam,
    jet_orientation_deg, trace_jet_ridgeline,
)
from radio_diffusion.data.preprocess_mojave import PREPROCESSING_SETTINGS
from radio_diffusion.paths import get_path


REAL_EXAMPLE_FILENAME = "0003+380.u.2007_08_24.icn.fits.gz"
REAL_EXAMPLE_SERIES = "estimate"


def read_mojave_example(filename, series):
    """Measure one native-beam FITS observation with the inspection settings."""
    path = get_path("MOJAVE_FITS_DIR") / filename
    orientation = JetOrientationSettings(inner_mas=1.0, outer_mas=3.0, sigma_cut=3.0)
    with fits.open(path, memmap=False) as hdul, warnings.catch_warnings():
        warnings.simplefilter("ignore", FITSFixedWarning)
        image_hdu = hdul[0]
        noise = get_noise_jy_per_beam(image_hdu, "four_corner")
        pa = jet_orientation_deg(image_hdu, hdul["AIPS CC"], noise, orientation)
        ridge = []
        if np.isfinite(pa):
            ridge, _ = trace_jet_ridgeline(image_hdu, noise, pa, PREPROCESSING_SETTINGS.ridgeline)
        frame = plotting.mojave_coordinates(crop_spatial_geometry(image_hdu.header, left=0, top=0))
        ridge_pixels = np.array([(point["x_pixel"], point["y_pixel"]) for point in ridge])
        return dict(
            id="R01",
            source=filename.split(".", 1)[0],
            image=np.asarray(np.squeeze(image_hdu.data), dtype=np.float32).copy(),
            coordinates=frame,
            ridge=frame.from_pixels(ridge_pixels) if ridge else np.empty((0, 2)),
            annotations={series: (pa if np.isfinite(pa) else None, ridge[-1]["radius_mas"] if ridge else None)},
        )


real_example = read_mojave_example(REAL_EXAMPLE_FILENAME, REAL_EXAMPLE_SERIES)
samples = [real_example, *synthetic_samples]

## What the caller supplies

The caller selects the fields and chooses the labels and styles. It turns valid angles and extents into marks and represents missing values with a dash in the same field position. The shared renderer receives the prepared marks and values without having to interpret the metadata.

In [ ]:
SHOW_AXES = True
SHARED_COLORBAR = True
SHOW_PLAIN_GRID = False
ANNOTATED_PANELS = None  # All panels, or supply a set of sample IDs.
VIEW_MAS = 3.7
INTENSITY_LIMITS = (-0.001, 0.56)
FADE_EXTENT_ARCS = True

SERIES = {
    "conditioning": SeriesStyle("#259BC4", "dashed", "Conditioning"),
    "estimate": SeriesStyle("#D47E16", "solid", "Image estimate"),
}
RIDGE_STYLE = SeriesStyle("#C98DE1")


def prepare_annotations(sample):
    if ANNOTATED_PANELS is not None and sample["id"] not in ANNOTATED_PANELS:
        return None, []

    marks = ImageOverlay(coordinates=sample["coordinates"])
    entries = []
    for key, style in SERIES.items():
        pa, extent = sample["annotations"].get(key, (None, None))
        pa = pa if pa is not None and np.isfinite(pa) else None
        extent = extent if extent is not None and np.isfinite(extent) and extent > 0 else None

        entries.append(KeyEntry("arrow", "Jet PA", "—" if pa is None else f"{pa:.0f}°", style))
        if pa is not None:
            angle = np.deg2rad(pa)
            tip = (1.65 * np.sin(angle), 1.65 * np.cos(angle))
            marks.append(ImageOverlay.arrow(
                (0, 0), tip, color=style.color, linestyle=style.linestyle, linewidth=1.25,
            ))

        entries.append(KeyEntry(
            "arc", "Radial extent", "—" if extent is None else f"{extent:.2f} mas",
            style, faded=FADE_EXTENT_ARCS,
        ))
        if pa is not None and extent is not None:
            marks.append(ImageOverlay.arc(
                (0, 0), extent, angle_deg=90 - pa, delta_angle_deg=24,
                fade=FADE_EXTENT_ARCS, color=style.color, linewidth=1.25,
                linestyle="solid" if FADE_EXTENT_ARCS else style.linestyle,
            ))

    ridge = sample["ridge"]
    if len(ridge) >= 2:
        marks.append(ImageOverlay.polyline(
            ridge, smooth=True, color=RIDGE_STYLE.color, linewidth=0.9, alpha=0.85,
        ))
    ridge_count = f"{len(ridge)} pts" if len(ridge) >= 2 else "—"
    entries.append(KeyEntry("polyline", "Ridgeline", ridge_count, RIDGE_STYLE))
    return marks, entries

In [ ]:
images = [sample["image"] for sample in samples]
overlays, tile_entries = zip(*(prepare_annotations(sample) for sample in samples))
grid_options = dict(
    nrow=4,
    sample_labels=[f"{sample['id']}  |  {sample['source']}" for sample in samples],
    coordinates=[sample["coordinates"] for sample in samples],
    show_axes=SHOW_AXES,
    shared_scale=SHARED_COLORBAR,
    limits=INTENSITY_LIMITS if SHARED_COLORBAR else None,
    colorbar_label="Jy/beam",
    view_limits=((VIEW_MAS, -VIEW_MAS), (-VIEW_MAS, VIEW_MAS)),
    figsize=(14, 9.9),
)


def display_grid(figure):
    png = BytesIO()
    figure.savefig(png, format="png", dpi=120)
    display(Image(data=png.getvalue()))


with mpl.rc_context({"font.size": 9, "axes.labelsize": 9, "figure.facecolor": "white"}):
    figure, _ = plot_keyed_grid(images, overlays=overlays, tile_entries=tile_entries, **grid_options)
    display_grid(figure)
    if SHOW_PLAIN_GRID:
        figure, _ = plot_keyed_grid(images, **grid_options)
        display_grid(figure)